# to do

create an MLP for the global features as exercises limb amplitude time some fft fearures and others 
concatanate it to the cNN

In [56]:
import sys 
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data" 



from python.dataset_creation import (unique_subjects, all_subjects, 
                                    X_glob,X_seq, y                                        
                                    )



from utilities.model import  CNNMLPModel



import torch
import numpy as np

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [ ]:
print(type(all_subjects))
print(np.array(all_subjects).shape)
print(all_subjects[:10] if hasattr(all_subjects, "__len__") else all_subjects)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# cross validation

In [ ]:
fold_results = []

for val_subject in unique_subjects:
    print(f"\n========== Fold: leaving out {val_subject} ==========")

    train_indices = [i for i, s in enumerate(all_subjects) if s != val_subject]
    val_indices = [i for i, s in enumerate(all_subjects) if s == val_subject]

    print(train_indices)

    X_seq_train = X_seq[train_indices]
    X_seq_val   = X_seq[val_indices]

    X_glob_train = X_glob[train_indices]
    X_glob_val   = X_glob[val_indices]

    y_train = y[train_indices]
    y_val   = y[val_indices]

    '''  train_mean = X_train.mean(dim=(0,2), keepdim=True)
    train_std = X_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_train = (X_train - train_mean) / train_std
    X_val = (X_val - train_mean) / train_std'''




    
    train_dataset = TensorDataset(
    X_seq_train,
    X_glob_train,
    y_train
)

    val_dataset = TensorDataset(
    X_seq_val,
    X_glob_val,
    y_val
)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    # fresh model for each fold
    model = CNNMLPModel(input_dim_seq=15,input_dim_global=9, num_classes=3).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_val_loss = float("inf")
    best_preds = None
    best_labels = None

    for epoch in range(20):
        # TRAIN
        model.train()
        running_loss = 0.0

        for x_seq_b, x_glob_b, y_b in train_loader:
            x_seq_b = x_seq_b.to(device)
            x_glob_b = x_glob_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()

            outputs = model(x_seq_b, x_glob_b)
            loss = criterion(outputs, y_b)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x_seq_b.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        # VALIDATION
        model.eval()
        val_running_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for x_seq_b, x_glob_b, y_b in val_loader:
                x_seq_b = x_seq_b.to(device)
                x_glob_b = x_glob_b.to(device)
                y_b = y_b.to(device)

                outputs = model(x_seq_b, x_glob_b)
                loss = criterion(outputs, y_b)

                val_running_loss += loss.item() * x_seq_b.size(0)

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_b.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)

        print(
            f"Fold {val_subject} | Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        # keep best epoch for this fold
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds = np.array(all_preds)
            best_labels = np.array(all_labels)

    # fold-level metrics
    fold_acc = accuracy_score(best_labels, best_preds)
    fold_f1 = f1_score(best_labels, best_preds, average="macro")
    fold_cm = confusion_matrix(best_labels, best_preds)

    fold_results.append({
        "subject": val_subject,
        "val_loss": best_val_loss,
        "accuracy": fold_acc,
        "macro_f1": fold_f1,
        "confusion_matrix": fold_cm
    })

    


========== Fold: leaving out s1 ==========
[240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 

KeyboardInterrupt: 

In [15]:
for r in fold_results:
    print(f"Subject: {r['subject']}")
    print(f"Val Loss: {r['val_loss']:.4f}")
    print(f"Accuracy: {r['accuracy']:.4f}")
    print(f"Macro F1: {r['macro_f1']:.4f}")
    print("Confusion Matrix:")
    print(r["confusion_matrix"])
    print("-" * 40)

Subject: s1
Val Loss: 0.5372
Accuracy: 0.6375
Macro F1: 0.6018
Confusion Matrix:
[[73  7  0]
 [67 13  0]
 [ 0 13 67]]
----------------------------------------
Subject: s2
Val Loss: 0.3353
Accuracy: 0.9083
Macro F1: 0.9073
Confusion Matrix:
[[73  7  0]
 [ 4 66 10]
 [ 0  1 79]]
----------------------------------------
Subject: s3
Val Loss: 0.4358
Accuracy: 0.8417
Macro F1: 0.8401
Confusion Matrix:
[[69 11  0]
 [27 53  0]
 [ 0  0 80]]
----------------------------------------
Subject: s4
Val Loss: 0.5415
Accuracy: 0.7000
Macro F1: 0.6374
Confusion Matrix:
[[11 69  0]
 [ 0 80  0]
 [ 0  3 77]]
----------------------------------------
Subject: s5
Val Loss: 0.3748
Accuracy: 0.8292
Macro F1: 0.8157
Confusion Matrix:
[[76  4  0]
 [25 43 12]
 [ 0  0 80]]
----------------------------------------


In [14]:
for s in np.unique(all_subjects):
    mask = np.array(all_subjects) == s
    print(s, np.bincount(y[mask]))

s1 [80 80 80]
s2 [80 80 80]
s3 [80 80 80]
s4 [80 80 80]
s5 [80 80 80]


# model training 


In [46]:
val_subject = 's2'   # change this to the patient you want to leave out

train_indices = [
    i for i, s in enumerate(all_subjects)
    if s != val_subject
]

val_indices = [
    i for i, s in enumerate(all_subjects)
    if s == val_subject
]

X_seq_train = X_seq[train_indices]
X_seq_val = X_seq[val_indices]

X_glob_train = X_glob[train_indices]
X_glob_val = X_glob[val_indices]

y_train = y[train_indices]
y_val = y[val_indices]

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Left-out subject:", val_subject)

Training samples: 960
Validation samples: 240
Left-out subject: s2


In [53]:
import copy
best_model_state = None

train_dataset = TensorDataset(
    X_seq_train,
    X_glob_train,
    y_train
)

val_dataset = TensorDataset(
    X_seq_val,
    X_glob_val,
    y_val
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# fresh model for each fold
model = CNNMLPModel(
    input_dim_seq=15,
    input_dim_global=9,
    num_classes=3
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_val_loss = float("inf")
best_preds = None
best_labels = None

for epoch in range(20):
    # TRAIN
    model.train()
    running_loss = 0.0

    for x_seq_b, x_glob_b, y_b in train_loader:
        x_seq_b = x_seq_b.to(device)
        x_glob_b = x_glob_b.to(device)
        y_b = y_b.to(device)

        optimizer.zero_grad()

        outputs = model(x_seq_b, x_glob_b)
        loss = criterion(outputs, y_b)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_seq_b.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # VALIDATION
    model.eval()
    val_running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_seq_b, x_glob_b, y_b in val_loader:
            x_seq_b = x_seq_b.to(device)
            x_glob_b = x_glob_b.to(device)
            y_b = y_b.to(device)

            outputs = model(x_seq_b, x_glob_b)
            loss = criterion(outputs, y_b)

            val_running_loss += loss.item() * x_seq_b.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())

    val_loss = val_running_loss / len(val_loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)

    print(
        f"Fold {val_subject} | Epoch {epoch + 1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # keep best epoch for this fold
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_preds = np.array(all_preds)
        best_labels = np.array(all_labels)
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

# fold-level metrics
fold_acc = accuracy_score(best_labels, best_preds)
fold_f1 = f1_score(best_labels, best_preds, average="macro")
fold_cm = confusion_matrix(best_labels, best_preds)

fold_results = {}

fold_results[val_subject] = {
    "subject": val_subject,
    "best_epoch": best_epoch,
    "val_loss": best_val_loss,
    "accuracy": fold_acc,
    "macro_f1": fold_f1,
    "confusion_matrix": fold_cm,
    "model_state_dict": best_model_state,
    "predictions": best_preds,
    "labels": best_labels
}

Fold s2 | Epoch 1 | Train Loss: 1.0107 | Val Loss: 0.9698 | Val Acc: 0.4583
Fold s2 | Epoch 2 | Train Loss: 0.8842 | Val Loss: 0.8999 | Val Acc: 0.5167
Fold s2 | Epoch 3 | Train Loss: 0.8212 | Val Loss: 0.8919 | Val Acc: 0.5167
Fold s2 | Epoch 4 | Train Loss: 0.7554 | Val Loss: 0.6756 | Val Acc: 0.7917
Fold s2 | Epoch 5 | Train Loss: 0.6859 | Val Loss: 0.6074 | Val Acc: 0.7958
Fold s2 | Epoch 6 | Train Loss: 0.6406 | Val Loss: 0.9643 | Val Acc: 0.4375
Fold s2 | Epoch 7 | Train Loss: 0.5981 | Val Loss: 1.0316 | Val Acc: 0.4250
Fold s2 | Epoch 8 | Train Loss: 0.5583 | Val Loss: 0.5195 | Val Acc: 0.7958
Fold s2 | Epoch 9 | Train Loss: 0.5472 | Val Loss: 0.4911 | Val Acc: 0.8250
Fold s2 | Epoch 10 | Train Loss: 0.5145 | Val Loss: 0.4840 | Val Acc: 0.8583
Fold s2 | Epoch 11 | Train Loss: 0.4898 | Val Loss: 1.0359 | Val Acc: 0.5125
Fold s2 | Epoch 12 | Train Loss: 0.4714 | Val Loss: 0.4884 | Val Acc: 0.8625
Fold s2 | Epoch 13 | Train Loss: 0.4379 | Val Loss: 0.4776 | Val Acc: 0.8625
Fold s2 

In [54]:
torch.save(fold_results, "fold_results.pth")